# Spatial proteomics biomarker workflows in Python
**scimap** (labsyspharm) for phenotyping, neighbourhoods, and interactions. **CytoCommunity** (supervised GNN niches) for condition-specific tissue neighbourhoods.

> Template. Install: `pip install scimap` · CytoCommunity: `git clone https://github.com/huBioinfo/CytoCommunity` (follow its conda env)

In [ ]:
import scimap as sm, anndata as ad, squidpy as sq
adata = sq.datasets.imc()
adata.obs["X_centroid"] = adata.obsm["spatial"][:, 0]
adata.obs["Y_centroid"] = adata.obsm["spatial"][:, 1]
adata.obs["imageid"] = "img1"
adata.obs["phenotype"] = adata.obs["cell type"].astype(str)

## Spatial distance, interaction, and neighbourhoods (scimap)

In [ ]:
adata = sm.tl.spatial_distance(adata, x_coordinate="X_centroid", y_coordinate="Y_centroid",
                               phenotype="phenotype", imageid="imageid")
adata = sm.tl.spatial_interaction(adata, x_coordinate="X_centroid", y_coordinate="Y_centroid",
                                  phenotype="phenotype", imageid="imageid", method="radius", radius=30)
sm.pl.spatial_interaction(adata, summarize_plot=True)

adata = sm.tl.spatial_count(adata, x_coordinate="X_centroid", y_coordinate="Y_centroid",
                            phenotype="phenotype", method="radius", radius=30, label="spatial_count")
adata = sm.tl.spatial_cluster(adata, df_name="spatial_count", method="kmeans", k=6, label="neighbourhood")
# Cellular neighbourhoods (Schurch et al. 2020 style). Patient-level neighbourhood frequencies are candidate biomarkers.
adata.obs.groupby(["imageid", "neighbourhood"]).size().unstack(fill_value=0)

## Neighbourhood-conditioned marker shifts (ctSVP flavour)
Within one cell type, compare a functional marker across neighbourhoods. A Kruskal–Wallis test per (cell type, marker) is a quick screen:

In [ ]:
import pandas as pd, numpy as np
from scipy.stats import kruskal
X = pd.DataFrame(adata.X, index=adata.obs_names, columns=adata.var_names)
res = []
for ct, d in adata.obs.groupby("phenotype"):
    groups = [g.index for _, g in d.groupby("neighbourhood") if len(g) >= 20]
    if len(groups) < 2: continue
    for m in X.columns:
        res.append((ct, m, kruskal(*[X.loc[g, m] for g in groups]).pvalue))
pd.DataFrame(res, columns=["cell_type", "feature", "p_value"]).sort_values("p_value").head(20)

## CytoCommunity (supervised, condition-specific niches)
CytoCommunity learns tissue cellular neighbourhoods that best separate patient conditions, e.g., CLR vs. DII in Schürch 2020 CODEX, or good vs. poor prognosis.
Its input is one folder per image with `*_Coordinates.txt`, `*_CellTypeLabel.txt`, and a `*_GraphLabel.txt` holding the condition.
```bash
git clone https://github.com/huBioinfo/CytoCommunity.git
cd CytoCommunity/Tutorial/Supervised   # step1: construct KNN graphs → step2: train TCN-learning GNN → step3: ensemble → step4: visualize
```
Our D8 (Schürch CODEX) single-cell CSV maps directly onto this format.